In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Dataset
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import imageio

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

#### 1) Dataset

In [ ]:
class StackedMNIST(Dataset):
    def __init__(self, mnist_dataset, num_samples=25600):
        self.num_samples = num_samples
        loader = DataLoader(mnist_dataset, batch_size=2048, shuffle=False)
        all_images = []
        all_labels = []

        for images, labels in loader:
            all_images.append(images)
            all_labels.append(labels)

        all_images = torch.cat(all_images, dim=0)
        all_labels = torch.cat(all_labels, dim=0)
        mnist_len = len(mnist_dataset)
        indices = torch.randint(0, mnist_len, (num_samples, 3))

        channel_0 = all_images[indices[:, 0]]
        channel_1 = all_images[indices[:, 1]]
        channel_2 = all_images[indices[:, 2]]
        self.data = torch.cat([channel_0, channel_1, channel_2], dim=1)

        lbl_0 = all_labels[indices[:, 0]]
        lbl_1 = all_labels[indices[:, 1]]
        lbl_2 = all_labels[indices[:, 2]]
        self.targets = lbl_0 * 100 + lbl_1 * 10 + lbl_2

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return self.data[idx], self.targets[idx]


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Pad(2),
    transforms.Normalize((0.5,), (0.5,))
])

basic_mnist = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
stacked_mnist_dataset = StackedMNIST(basic_mnist, num_samples=25600)
dataloader = DataLoader(stacked_mnist_dataset, batch_size=128, shuffle=True)

#### 2) Architecture

In [ ]:
class Generators(nn.Module):
    def __init__(self, z_dim, num_g):
        super(Generators, self).__init__()
        self.num_generators = num_g

        self.shared_layers = nn.Sequential(
            nn.Linear(z_dim, 4 * 4 * 64),
            nn.ReLU(True),
            nn.Unflatten(1, (64, 4, 4)),

            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1, bias=False),
            nn.ReLU(True),

            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1, bias=False),
            nn.ReLU(True),
        )

        self.unshared_layers = nn.ModuleList([
            nn.Sequential(
                nn.ConvTranspose2d(16, 8, kernel_size=4, stride=2, padding=1, bias=False),
                nn.ReLU(True),

                nn.Conv2d(8, 3, kernel_size=3, stride=1, padding=1, bias=False),  # 3 канала
                nn.Tanh()
            ) for _ in range(self.num_generators)
        ])

    def forward_single(self, z, gen_idx):
        shared_features = self.shared_layers(z)
        fake_images = self.unshared_layers[gen_idx](shared_features)
        return fake_images

    def forward_all(self, z_list):
        fake_images_list = []
        for num in range(self.num_generators):
            fake_img = self.forward_single(z_list[num], gen_idx=num)
            fake_images_list.append(fake_img)
        return fake_images_list


class Discriminator(nn.Module):
    def __init__(self, num_g):
        super(Discriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 4, kernel_size=4, stride=2, padding=1, bias=False),  # 3 канала
            nn.LeakyReLU(0.3),

            nn.Conv2d(4, 8, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.3),

            nn.Conv2d(8, 16, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.3),

            nn.Flatten(),
            nn.Linear(256, num_g + 1),
        )

    def forward(self, x):
        return self.net(x)

#### 3) Classifier and metrics

In [ ]:
def weights_init_xavier(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1 or classname.find('Linear') != -1:
        nn.init.xavier_normal_(m.weight.data)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0.0)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0.0)


def get_pretrained_resnet():
    model = models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model


def classify_images(classifier, fake_images):
    classifier.eval()
    mode_indices = 0
    with torch.no_grad():
        for channel in range(3):
            image_channel = fake_images[:, channel:channel+1, :, :]
            logits = classifier(image_channel)
            prediction = logits.argmax(dim=1).cpu().numpy()
            mode_indices += prediction * 10 ** (2 - channel)
    return mode_indices


def compute_metrics(classifier, generator, num_samples=25600, b_size=64, l_dim=256):
    generator.eval()
    classifier.eval()

    all_mode_indices = []
    num_generators = generator.num_generators
    num_batches = num_samples // (b_size * num_generators) + 1

    with torch.no_grad():
        for _ in range(num_batches):
            z_noise_list = [
                torch.randn(b_size, l_dim, device=device)
                for _ in range(num_generators)
            ]
            fake_list = torch.cat(generator.forward_all(z_noise_list), dim=0)
            modes = classify_images(classifier, fake_list)
            all_mode_indices.extend(modes)

    all_mode_indices = np.array(all_mode_indices[:num_samples], dtype=int)

    counts = np.bincount(all_mode_indices, minlength=1000)
    number_of_modes = np.count_nonzero(counts)

    P = np.full(1000, 1.0 / 1000)
    epsilon = 1e-8
    Q = (counts / len(all_mode_indices)) + epsilon
    Q = Q / np.sum(Q)

    kl_divergence = np.sum(P * np.log(P / Q))

    return number_of_modes, kl_divergence


#Load the classifier
classifier = get_pretrained_resnet().to(device)
classifier.load_state_dict(torch.load('/kaggle/input/datasets/username/classifier-mnist-stacked/mnist_classifier.pth'))
classifier.eval()

#History of training
history = {
    'loss_d': [],
    'loss_g': [],
    'modes_count': [],
    'kl_div': [],
    'epochs_checked': []
}

best_modes = 0
best_kl = 1e10
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('samples', exist_ok=True)

gif_frames = []

#### 4) Initialization

In [ ]:
latent_dim = 256
lr = 0.0002
batch_size = 128
epochs = 5000
num_gens = 3
lambda_prox = 0.1
T = 3

generators = Generators(z_dim=latent_dim, num_g=num_gens).to(device)
discriminator = Discriminator(num_g=num_gens).to(device)

generators.apply(weights_init_xavier)
discriminator.apply(weights_init_xavier)

optimizer_G = optim.Adam(generators.parameters(), lr=lr, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))

loss_game = nn.CrossEntropyLoss()
fixed_noise = torch.randn(6, latent_dim, device=device)

def compute_grad(D_weights, real_data):
    real_data.requires_grad_(True)

    grad = torch.autograd.grad(
        outputs=D_weights(real_data).sum(),
        inputs=real_data,
        create_graph=True
    )[0]
    return grad

#### 5) Proximal-training

In [ ]:
for epoch in range(epochs):
    generators.train()
    discriminator.train()

    for i, (real_imgs, _) in enumerate(dataloader):
        batch_size_curr = real_imgs.size(0)
        real_imgs = real_imgs.to(device)

        # Discriminator
        D_old = compute_grad(discriminator, real_imgs).detach()

        for t in range(T):
            optimizer_D.zero_grad()

            real_labels = torch.full((batch_size_curr,), num_gens, dtype=torch.long, device=device)
            output_real = discriminator(real_imgs)
            loss_D_real = loss_game(output_real, real_labels)

            loss_D_fake = torch.tensor(0.0, device=device)

            for j in range(num_gens):
                noise = torch.randn(batch_size_curr, latent_dim, device=device)
                fake_imgs = generators.forward_single(z=noise, gen_idx=j)
                fake_labels = torch.full((batch_size_curr,), j, dtype=torch.long, device=device)
                output_fake = discriminator(fake_imgs.detach())
                loss_D_fake += loss_game(output_fake, fake_labels)

            D_new = compute_grad(discriminator, real_imgs)
            proximal_penalty = torch.mean((D_new - D_old).pow(2).sum(dim=1))

            loss_D = loss_D_real + loss_D_fake + lambda_prox*proximal_penalty
            loss_D.backward()
            optimizer_D.step()

        # Generators
        optimizer_G.zero_grad()

        loss_G_total = torch.tensor(0.0, device=device)

        for j in range(num_gens):
            noise = torch.randn(batch_size_curr, latent_dim, device=device)
            fake_imgs = generators.forward_single(z=noise, gen_idx=j)
            output_fake = discriminator(fake_imgs)
            loss_G = loss_game(output_fake, real_labels)
            loss_G_total += loss_G

        loss_G_total.backward()
        optimizer_G.step()

    print(f"Epoch [{epoch+1}/{epochs}] | Loss D: {loss_D.item():.4f} | Loss G: {loss_G.item():.4f}")
    history['loss_d'].append(loss_D.item())
    history['loss_g'].append(loss_G.item())

    if (epoch + 1) % 100 == 0 or (epoch + 1) == epochs:
        generators.eval()
        with torch.no_grad():
            z_shared = fixed_noise
            fake_images_list = []
            for gen_idx in range(num_gens):
                imgs = generators.forward_single(z_shared, gen_idx)
                fake_images_list.append(imgs)
    
            combined = torch.cat(fake_images_list, dim=0)
    
            channels = []
            for c in range(3):
                channel = combined[:, c:c+1, :, :]
                grid_c = vutils.make_grid(channel, nrow=6, padding=2, normalize=True)
                channels.append(grid_c)
    
            full_grid = torch.cat(channels, dim=1)
    
            # Save image
            vutils.save_image(full_grid, f'samples/epoch_{epoch+1:04d}.png')
            ndarr = full_grid.mul(255).add_(0.5).clamp_(0, 255)
            ndarr = ndarr.permute(1, 2, 0).to('cpu', torch.uint8).numpy()
            gif_frames.append(Image.fromarray(ndarr))

            plt.figure(figsize=(12, 8))
            plt.imshow(full_grid.cpu().permute(1, 2, 0).squeeze(), cmap='gray')
            plt.title(f"Epoch {epoch+1}")
            plt.axis('off')
            plt.show()
    
    # Metrics
    if (epoch + 1) % 100 == 0 or (epoch + 1) == epochs:
        modes_count, kl_div = compute_metrics(
            classifier=classifier,
            generator=generators,
            num_samples=25600,
            b_size=64,
            l_dim=latent_dim
        )

        history['modes_count'].append(modes_count)
        history['kl_div'].append(kl_div)
        history['epochs_checked'].append(epoch + 1)

        print(f"Epoch {epoch+1} | Modes: {modes_count}/1000 | KL: {kl_div:.4f}")

    # Save weights
    if (epoch + 1) % 1000 == 0:
        best_modes = max(best_modes, modes_count)
        best_kl = min(best_kl, kl_div)
        torch.save({
            'generators': generators.state_dict(),
            'discriminator': discriminator.state_dict(),
            'epoch': epoch + 1,
            'modes': modes_count,
            'kl': kl_div
        }, f'checkpoints/best_madgan_modes{modes_count}_kl{kl_div:.4f}.pth')
        print(f"Saved weights (modes={modes_count}, kl={kl_div:.4f})")

In [ ]:
torch.save({
    'generators': generators.state_dict(),
    'discriminator': discriminator.state_dict(),
    'history': history
}, 'checkpoints/final_madgan.pth')

torch.save(history, 'checkpoints/history.pt')
print("History saved")

if gif_frames:
    gif_frames[0].save(
        'samples/training.gif',
        save_all=True,
        append_images=gif_frames[1:],
        duration=300,
        loop=0
    )
    print("GIF saved")